# ReconMind Stage 3, Model B: fine tuned SciBERT ATT&CK mapper (training run 3)

This notebook trains Model B again, this time on the run 3 training data, and then scores it in two places: the TRAM eval set (the same one Model A and run 2 were scored on) and the ReconMind test set of real scan findings.

**Why run 3.** Run 2 learned only from TRAM's threat report sentences and predicted nothing on ReconMind's findings. Run 3 trains on data that looks like those findings: ATT&CK procedure examples, MITRE CTID CVE mappings, Exploit-DB titles written the way ReconMind writes a finding, and TRAM's own train and val splits. The data is built on your computer by `evaluation/scripts/prepare_run3_data.py` and read here from your Google Drive.

**What changed since run 2.** Only these things:

1. **Training data and label set.** Training and validation sentences come from the run 3 data, and the model has one output per technique in run 3's `labels.json`. The count is read from that file, not typed in.
2. **At most 12 epochs instead of 15.** Each epoch has more than twice as many sentences as in run 2. Early stopping still uses a patience of 3.
3. **Two TRAM scorings.** Run 3 can predict techniques outside TRAM's 50. The eval set is predicted once and scored twice: with predictions limited to TRAM's 50 (`tram50_filtered`, directly comparable to run 2 and Model A) and with every prediction kept (`unfiltered`, where an ID outside TRAM's 50 counts as out of set, a wrong answer).
4. **ReconMind test set predictions.** The trained model also predicts every ReconMind finding, so no separate prediction notebook is needed for run 3.

The seed, the learning rate and its schedule, the rare technique weights and their cap, the batch sizes, the plain SciBERT starting point, the maximum length rule, choosing the epoch and threshold on validation only, the eval fingerprint check and the scoring code are all the same as run 2.

**Before you run it:**

1. On your computer, run `python3 evaluation/scripts/prepare_run3_data.py --exploitdb-csv PATH`, commit `evaluation/results/run3_data_manifest.json` and push the `attack-mapper` branch. The notebook clones that branch and checks the data against this manifest.
2. Upload `train.jsonl`, `val.jsonl` and `labels.json` from `evaluation/data/run3/` to the Drive folder `MyDrive/ReconMind/run3_data/`.

**How to run:** use a T4 GPU, then choose Runtime, Run all. Google asks for permission to use your Drive near the start. Run 2 took about 2 minutes 15 seconds per epoch on 13,143 training sentences. Run 3 has 29,438, so allow about 5 minutes per epoch and up to about an hour if all 12 epochs run. At the end your browser downloads the metrics file and the ReconMind predictions.

## 1. Run settings

Every setting is in one place and is written to the metrics file. Only `RUN_NAME` and `EPOCHS` differ from run 2.

- `RUN_NAME` goes into the name of every output file and the Drive folder, so run 1 and run 2 files are never overwritten.
- `EPOCHS` is the most passes over the training data. Early stopping may end training sooner.
- `PATIENCE` is how many epochs in a row may pass without a better validation micro F1 before training stops.
- `LEARNING_RATE` is the peak learning rate, reached at the end of warmup.
- `WARMUP_FRACTION` is the share of all planned training steps spent warming up.
- `POS_WEIGHT_CAP` is the largest extra weight any single technique can get in the loss.

In [ ]:
RUN_NAME = 'run3'
EPOCHS = 12
PATIENCE = 3
LEARNING_RATE = 3e-5
WARMUP_FRACTION = 0.1
POS_WEIGHT_CAP = 30.0

assert RUN_NAME not in ('run1', 'run2'), 'choose a new RUN_NAME so earlier results are kept'
print('run:', RUN_NAME, '| epochs up to', EPOCHS, '| patience', PATIENCE,
      '| lr', LEARNING_RATE, '| warmup', WARMUP_FRACTION, '| pos_weight cap', POS_WEIGHT_CAP)

## 2. Check for a GPU

Training on a CPU would take many hours, so this cell stops right away if no GPU is attached and tells you how to add one.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU is attached. In the Colab menu choose Runtime, Change runtime type, '
        'pick T4 GPU, click Save, then choose Runtime, Run all.')

GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
if 'T4' not in GPU_NAME:
    print('Note: this notebook was written for a T4. Other GPUs work, timings will differ.')

## 3. Get the ReconMind code

Downloads the project from GitHub (branch `attack-mapper`) so we can reuse its data preparation and scoring code instead of copying it here. It also brings `evaluation/results/run3_data_manifest.json`, which section 4 checks the training data against. The commit ID is recorded so the result can be traced to the exact code that produced it.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/r00tSud0/ReconMind'
BRANCH = 'attack-mapper'
REPO_DIR = '/content/ReconMind'

if os.path.isdir(REPO_DIR):
    print('Already cloned, reusing', REPO_DIR)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
                   check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

CODE_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], check=True,
                             capture_output=True, text=True).stdout.strip()
print('code commit:', CODE_COMMIT)

## 4. Connect Google Drive and check the run 3 data

Connects your Drive now, so any problem shows up before training starts. A popup asks for permission.

Reads `train.jsonl`, `val.jsonl` and `labels.json` from `MyDrive/ReconMind/run3_data/` as raw bytes and computes a fingerprint (SHA-256) of each one. These are compared with the `outputs` section of `evaluation/results/run3_data_manifest.json` in the cloned code. If a file is missing, the manifest has no fingerprint for it, or any fingerprint differs, the notebook stops: the data in Drive is not the data the manifest describes.

Later cells read the sentences from these same bytes, so the data that was checked is exactly the data that is trained on.

In [ ]:
import hashlib
import json

from google.colab import drive

drive.mount('/content/drive')
RUN3_DATA_DIR = '/content/drive/MyDrive/ReconMind/run3_data'
RUN3_FILES = ('train.jsonl', 'val.jsonl', 'labels.json')
MANIFEST_PATH = os.path.join(REPO_DIR, 'evaluation', 'results', 'run3_data_manifest.json')

with open(MANIFEST_PATH) as fh:
    EXPECTED_SHA256 = json.load(fh).get('outputs', {})

run3_raw, RUN3_SHA256, problems = {}, {}, []
for name in RUN3_FILES:
    path = os.path.join(RUN3_DATA_DIR, name)
    if not os.path.isfile(path):
        problems.append('{n} is not in {d}'.format(n=name, d=RUN3_DATA_DIR))
        continue
    with open(path, 'rb') as fh:
        run3_raw[name] = fh.read()
    RUN3_SHA256[name] = hashlib.sha256(run3_raw[name]).hexdigest()
    expected = EXPECTED_SHA256.get(name)
    if expected is None:
        problems.append('the manifest has no SHA-256 for {n}; run prepare_run3_data.py again, '
                        'commit the manifest and push'.format(n=name))
    elif RUN3_SHA256[name] != expected:
        problems.append('{n} has SHA-256 {a}, the manifest says {e}'.format(
            n=name, a=RUN3_SHA256[name], e=expected))
    else:
        print('OK  {n}  {s}'.format(n=name, s=RUN3_SHA256[name]))

if problems:
    raise RuntimeError('The run 3 data in Drive does not match ' + MANIFEST_PATH + ': '
                       + '; '.join(problems) + '. Stopping before training.')
print('All three files match the manifest.')

## 5. Install the Hugging Face library

`transformers` provides SciBERT, its tokenizer and the learning rate schedule. PyTorch already comes with Colab. The versions are printed and saved with the results.

In [ ]:
!pip install -q transformers

import transformers

print('torch', torch.__version__, '| transformers', transformers.__version__)

## 6. Prepare the TRAM data

Runs the same script Model A and run 2 used. It downloads MITRE's TRAM sentences pinned to one commit, checks the file's checksum, and writes the splits and TRAM's 50 technique IDs to `evaluation/data/tram/`. Run 3 uses only TRAM's eval set and its 50 IDs from here; TRAM's train and val sentences are already inside the run 3 data. If anything is wrong it fails here and Run all stops.

In [ ]:
subprocess.run([sys.executable, 'evaluation/scripts/prepare_tram_data.py'], check=True)

## 7. Confirm the eval set is the one Model A was scored on

A fingerprint (SHA-256) of `eval.jsonl` is computed exactly the way `evaluation/scripts/eval_prompted.py` computes `eval_sha256`: over the raw file bytes. If it does not match the known value, the models would not be comparable, so the notebook stops.

In [ ]:
EXPECTED_EVAL_SHA256 = '46860e84293aaccc1d7fdc1094e036dd4c4d4a717d432eb6493cbe08eb20445c'
TRAM_DIR = os.path.join(REPO_DIR, 'evaluation', 'data', 'tram')

# Same steps as _load_eval in evaluation/scripts/eval_prompted.py.
with open(os.path.join(TRAM_DIR, 'eval.jsonl'), 'rb') as fh:
    eval_raw = fh.read()
EVAL_SHA256 = hashlib.sha256(eval_raw).hexdigest()

print('eval fingerprint:', EVAL_SHA256)
if EVAL_SHA256 != EXPECTED_EVAL_SHA256:
    raise RuntimeError('The eval set fingerprint does not match the one Model A used '
                       '(expected ' + EXPECTED_EVAL_SHA256 + '). Stopping so the '
                       'results stay comparable.')
print('OK: same eval set as Model A.')

## 8. Load the sentences and the labels

The run 3 train and validation sentences are read from the bytes checked in section 4. The label list is read from run 3's `labels.json`, and its length sets how many outputs the model has. Each training and validation sentence's labels become a row of zeros and ones, one per label (a "multi hot" vector), which is the target the model learns to predict.

TRAM's eval sentences and TRAM's 50 IDs come from section 6. A few of TRAM's 50 IDs may be missing from the run 3 label set, because `prepare_run3_data.py` drops IDs that are revoked or deprecated in ATT&CK v20.0. The model can never predict those, so every eval sentence that has one counts it as a miss. They are printed here and saved in the metrics file. Eval sentences get no target vector, since they are only scored against TRAM's 50 with the scoring code.

In [ ]:
from evaluation.attack_mapper import finetune_utils, metrics


def parse_jsonl(raw):
    return [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()]


train_rows = parse_jsonl(run3_raw['train.jsonl'])
val_rows = parse_jsonl(run3_raw['val.jsonl'])
eval_rows = parse_jsonl(eval_raw)

LABELS = [t['id'] for t in json.loads(run3_raw['labels.json'].decode('utf-8'))]
N_LABELS = len(LABELS)
assert N_LABELS > 0 and len(set(LABELS)) == N_LABELS, 'labels.json must list unique IDs'

with open(os.path.join(TRAM_DIR, 'labels.json')) as fh:
    TRAM_IDS = [t['id'] for t in json.load(fh)]
TRAM_SET = set(TRAM_IDS)
TRAM_NOT_IN_LABELS = sorted(TRAM_SET - set(LABELS))

print('run 3 labels:', N_LABELS)
print('TRAM IDs: {n}, not in the run 3 label set: {m}'.format(
    n=len(TRAM_IDS), m=', '.join(TRAM_NOT_IN_LABELS) or 'none'))
for name, rows in (('train', train_rows), ('val', val_rows), ('eval', eval_rows)):
    labeled = sum(1 for r in rows if r['labels'])
    print('{n:<5} {s:>6} sentences, {l:>5} with at least one technique'.format(
        n=name, s=len(rows), l=labeled))

## 9. Fix the random seed

Setting seed 42 everywhere makes the run repeatable: the same shuffling and the same starting weights for the new output layer. This is the same seed as runs 1 and 2. Training in half precision (fp16) on a GPU is close to, but not bit for bit, reproducible, so a rerun may differ in the last decimals.

In [ ]:
import random

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)
print('seed', SEED)

## 10. Load the tokenizer and pick the maximum sentence length

We start from plain SciBERT (`allenai/scibert_scivocab_uncased`), a general science model, and never from MITRE's own TRAM fine tuned checkpoint, which has already seen TRAM's data. Run 3 starts from plain SciBERT again, not from run 1's or run 2's model.

The tokenizer splits sentences into tokens. Longer limits are slower, so we use 256 tokens unless more than 1 percent of training sentences would be cut off, in which case we use 512. This is the same rule as run 2, applied to the run 3 training sentences.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'allenai/scibert_scivocab_uncased'
assert 'tram' not in MODEL_NAME.lower(), 'start from plain SciBERT, not a TRAM checkpoint'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Untruncated lengths, counting the [CLS] and [SEP] tokens the model adds.
train_lengths = [len(ids) for ids in tokenizer([r['sentence'] for r in train_rows])['input_ids']]
MAX_LENGTH, CUT_AT_256 = finetune_utils.choose_max_length(train_lengths)

print('{p:.2f}% of training sentences are longer than 256 tokens'.format(p=100 * CUT_AT_256))
if MAX_LENGTH == 512:
    cut_512 = sum(1 for n in train_lengths if n > 512) / len(train_lengths)
    print('{p:.2f}% are longer than 512 tokens'.format(p=100 * cut_512))
print('max length:', MAX_LENGTH)

## 11. Turn the sentences into batches

Tokenizes every sentence once and groups them into batches. Training batches hold 16 sentences and are shuffled (with the fixed seed). Validation and eval batches are larger and keep their order, since no learning happens there. Each batch is padded only to its own longest sentence, which saves time.

One difference from run 2: a set of sentences can be built without targets. TRAM's eval set and the ReconMind findings are built that way, because their gold labels are not needed to predict and some of TRAM's may be outside the run 3 label set. Their targets are rows of zeros that are never used.

The training set is kept in its own variable (`train_set`) because section 12 counts its labels to set the rare technique weights.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import DataCollatorWithPadding

BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64


class SentenceSet(Dataset):
    def __init__(self, texts, label_lists=None):
        self.enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH)
        if label_lists is None:
            # Prediction only: placeholder targets, never used for learning or scoring.
            self.targets = [[0] * N_LABELS for _ in texts]
        else:
            self.targets = [finetune_utils.to_multi_hot(labs, LABELS) for labs in label_lists]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.enc.items()}
        item['labels'] = self.targets[i]
        return item


pad = DataCollatorWithPadding(tokenizer)


def collate(items):
    labels = torch.tensor([item.pop('labels') for item in items], dtype=torch.float)
    batch = pad(items)
    batch['labels'] = labels
    return batch


train_set = SentenceSet([r['sentence'] for r in train_rows], [r['labels'] for r in train_rows])
shuffle_gen = torch.Generator()
shuffle_gen.manual_seed(SEED)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          generator=shuffle_gen, collate_fn=collate)
val_loader = DataLoader(SentenceSet([r['sentence'] for r in val_rows],
                                    [r['labels'] for r in val_rows]),
                        batch_size=EVAL_BATCH_SIZE, collate_fn=collate)
eval_loader = DataLoader(SentenceSet([r['sentence'] for r in eval_rows]),
                         batch_size=EVAL_BATCH_SIZE, collate_fn=collate)
print('training batches per epoch:', len(train_loader))

## 12. Load SciBERT with one output per run 3 label and a weighted loss

Adds a fresh output layer with one score per technique in run 3's `labels.json`. Each score is judged on its own (yes or no), which is what "multi label" means. The loss is binary cross entropy on the raw scores (BCE with logits). A warning that some weights are "newly initialized" is expected: that is the new output layer.

Rare techniques get extra weight in the loss, exactly as in run 2. Each technique's weight (`pos_weight`) is the number of training sentences without it divided by the number with it, capped at `POS_WEIGHT_CAP` (30). A technique with no training examples also gets the cap. The counts come from the train split only, so nothing about validation or eval leaks into training.

Because the training data and label set differ, run 3's training loss cannot be compared with run 2's. Compare runs on the eval scores instead.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_LABELS,
    problem_type='multi_label_classification',
    id2label=dict(enumerate(LABELS)),
    label2id={lab: i for i, lab in enumerate(LABELS)},
).to('cuda')

# negatives / positives per technique, counted on the TRAIN split only, capped.
POS_WEIGHT = finetune_utils.compute_pos_weight(train_set.targets, POS_WEIGHT_CAP)
loss_fn = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(POS_WEIGHT, dtype=torch.float, device='cuda'))

n_capped = sum(1 for w in POS_WEIGHT if w >= POS_WEIGHT_CAP)
print('pos_weight: min {lo:.2f}, median {md:.2f}, max {hi:.2f}'.format(
    lo=min(POS_WEIGHT), md=float(np.median(POS_WEIGHT)), hi=max(POS_WEIGHT)))
print('{n} of {t} techniques are at the cap of {c}'.format(
    n=n_capped, t=N_LABELS, c=POS_WEIGHT_CAP))

## 13. A helper that turns sentences into probabilities

Runs the model without learning and squashes each score into a probability between 0 and 1 (the sigmoid function). Used for validation, once for the TRAM eval set, and once for the ReconMind findings.

In [ ]:
@torch.no_grad()
def predict_probs(loader):
    model.eval()
    probs = []
    for batch in loader:
        inputs = {k: v.to('cuda') for k, v in batch.items() if k != 'labels'}
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(**inputs).logits
        probs.extend(torch.sigmoid(logits.float()).cpu().tolist())
    return probs

## 14. Train

Up to `EPOCHS` (12) passes over the run 3 training sentences with the AdamW optimizer, in half precision (fp16) for speed. After each epoch the model is checked on run 3's validation split only: we try thresholds from 0.05 to 0.95 and take the best micro F1 over the run 3 labels. The epoch with the best validation score is kept (an earlier epoch wins a tie). Neither the TRAM eval set nor the ReconMind findings are touched here.

The learning rate climbs in a straight line from zero to the peak of 3e-5 over the first 10 percent of all planned steps, then falls in a straight line to zero at the end of epoch 12, as in run 2. Training stops early once validation micro F1 has failed to beat its best so far for `PATIENCE` (3) epochs in a row. A tie does not count as an improvement. The best epoch's weights are saved as training goes and restored in section 15.

The learning rate at the end of each epoch is printed and saved with the epoch history.

In [ ]:
import time

from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
TOTAL_STEPS = EPOCHS * len(train_loader)
WARMUP_STEPS = int(WARMUP_FRACTION * TOTAL_STEPS)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=TOTAL_STEPS)
scaler = torch.amp.GradScaler('cuda')
val_gold = [r['labels'] for r in val_rows]
print('{t} training steps planned, the first {w} are warmup'.format(
    t=TOTAL_STEPS, w=WARMUP_STEPS))

best_f1, best_epoch, best_state = -1.0, None, None
epochs_since_best = 0
STOPPED_EARLY = False
epoch_history = []
train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start = time.time()
    loss_sum, steps = 0.0, 0
    for batch in train_loader:
        batch = {k: v.to('cuda') for k, v in batch.items()}
        targets = batch.pop('labels')
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_sum += loss.item()
        steps += 1

    val_probs = predict_probs(val_loader)
    threshold, val_f1, _ = finetune_utils.sweep_threshold(val_gold, val_probs, LABELS)
    val_f1_at_half = metrics.score(
        val_gold, finetune_utils.predict_all(val_probs, LABELS, 0.5), LABELS)['micro']['f1']
    lr_now = scheduler.get_last_lr()[0]
    row = {'epoch': epoch, 'train_loss': loss_sum / steps, 'val_micro_f1': val_f1,
           'val_threshold': threshold, 'val_micro_f1_at_0.5': val_f1_at_half,
           'lr_at_epoch_end': lr_now, 'seconds': round(time.time() - epoch_start, 1)}
    epoch_history.append(row)

    kept = val_f1 > best_f1
    if kept:
        best_f1, best_epoch = val_f1, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_since_best = 0
    else:
        epochs_since_best += 1
    print('epoch {e}: loss {l:.4f}  val micro F1 {f:.3f} at threshold {t}  '
          '(F1 at 0.5: {h:.3f})  lr {r:.2e}  {s:.0f}s{k}'.format(
              e=epoch, l=row['train_loss'], f=val_f1, t=threshold, h=val_f1_at_half,
              r=lr_now, s=row['seconds'], k='  (best so far)' if kept else ''))

    if epochs_since_best >= PATIENCE:
        STOPPED_EARLY = True
        print('No validation improvement for {p} epochs, stopping early.'.format(p=PATIENCE))
        break

EPOCHS_RUN = len(epoch_history)
TRAIN_SECONDS = round(time.time() - train_start, 1)
print('epochs run: {n} of {m} | best epoch: {b} | training time: {t:.1f} min'.format(
    n=EPOCHS_RUN, m=EPOCHS, b=best_epoch, t=TRAIN_SECONDS / 60))

## 15. Restore the best epoch and pick the threshold

Loads the weights from the best epoch back into the model, which matters when early stopping ran a few epochs past it. A technique is predicted when its probability is at least the threshold, and one threshold is used for every technique. It is chosen on run 3's validation split only (best micro F1, lowest threshold on ties), never on the TRAM eval set or the ReconMind findings. This is the same rule as run 2.

In [ ]:
model.load_state_dict(best_state)
val_probs = predict_probs(val_loader)
THRESHOLD, VAL_F1, val_sweep = finetune_utils.sweep_threshold(val_gold, val_probs, LABELS)

print('threshold   P      R      F1  (validation)')
for row in val_sweep:
    print('  {t:.2f}    {p:.3f}  {r:.3f}  {f:.3f}{m}'.format(
        t=row['threshold'], p=row['precision'], r=row['recall'], f=row['f1'],
        m='  (chosen)' if row['threshold'] == THRESHOLD else ''))

## 16. Score the TRAM eval set, once, two ways

Predicts the TRAM eval set a single time with the chosen epoch and threshold. The same predictions are then scored twice with `evaluation/attack_mapper/metrics.py`, the code that scored Model A and run 2. Both scorings use TRAM's 50 IDs as the label set:

- `tram50_filtered`: each sentence's predictions are limited to TRAM's 50 IDs before scoring. This is directly comparable to run 2 and Model A, which could only answer with those 50.
- `unfiltered`: every prediction is kept, so an ID outside TRAM's 50 counts as out of set, which is a false positive in micro precision and breaks an exact match.

In [ ]:
eval_probs = predict_probs(eval_loader)
eval_preds = finetune_utils.predict_all(eval_probs, LABELS, THRESHOLD)

gold_by_id = {r['id']: r['labels'] for r in eval_rows}
pred_unfiltered = {r['id']: p for r, p in zip(eval_rows, eval_preds)}
pred_filtered = {i: [lab for lab in p if lab in TRAM_SET] for i, p in pred_unfiltered.items()}
results = {
    'tram50_filtered': metrics.score(gold_by_id, pred_filtered, TRAM_IDS),
    'unfiltered': metrics.score(gold_by_id, pred_unfiltered, TRAM_IDS),
}

first = results['tram50_filtered']
print('sentences: {n} ({l} labeled, {u} unlabeled)'.format(
    n=first['n_sentences'], l=first['n_labeled'], u=first['n_unlabeled']))
for name, res in results.items():
    mi, ma = res['micro'], res['macro']
    print()
    print(name)
    print('  micro:        P {p:.3f}  R {r:.3f}  F1 {f:.3f}'.format(
        p=mi['precision'], r=mi['recall'], f=mi['f1']))
    print('  macro F1:     {f:.3f} over {n} techniques'.format(
        f=ma['f1'], n=ma['n_techniques_averaged']))
    print('  exact match:  {e:.3f}'.format(e=res['exact_match']))
    print('  out of set:   {n} predicted IDs'.format(n=res['out_of_set']['count']))
if first['zero_support_techniques']:
    print()
    print('zero support: {z}'.format(z=', '.join(first['zero_support_techniques'])))

## 17. Predict the ReconMind test set

Reads `evaluation/data/reconmind_eval/findings.jsonl` from the cloned code. Each line is one open port from a real scan, and only its `text` field (the plain English description ReconMind writes) goes to the model, the same input Model A gets. The model uses the epoch, threshold and maximum length chosen above, so the findings never influence any setting.

The answer key is never opened here. Scoring happens on your computer with `evaluation/scripts/score_reconmind_exam.py --model-b-run run3`.

In [ ]:
FINDINGS_PATH = os.path.join(REPO_DIR, 'evaluation', 'data', 'reconmind_eval', 'findings.jsonl')
with open(FINDINGS_PATH, 'rb') as fh:
    findings_raw = fh.read()
FINDINGS_SHA256 = hashlib.sha256(findings_raw).hexdigest()
findings = parse_jsonl(findings_raw)

exam_loader = DataLoader(SentenceSet([f['text'] for f in findings]),
                         batch_size=EVAL_BATCH_SIZE, collate_fn=collate)
exam_probs = predict_probs(exam_loader)
exam_preds = finetune_utils.predict_all(exam_probs, LABELS, THRESHOLD)

print(len(findings), 'findings | findings fingerprint', FINDINGS_SHA256)
for f, pred in zip(findings, exam_preds):
    print('{i:<22} {p}'.format(i=f['id'], p=', '.join(pred) if pred else '(none)'))

## 18. Save the results

Writes three files, all named from `RUN_NAME`, so earlier runs' files are never overwritten:

- `evaluation/results/finetuned_scibert_run3_metrics.json`: everything run 2 records (settings, best epoch, threshold, max length, timings, seed, library versions, epoch history, threshold sweep, `pos_weight` per technique) plus both TRAM scorings under `metrics`, the number of labels, the TRAM IDs missing from the label set, the fingerprints of the training data, and the ReconMind findings fingerprint.
- `evaluation/results/raw/finetuned_scibert_run3.jsonl`: one line per TRAM eval sentence with the gold labels, all predicted labels (unfiltered) and every probability in `labels.json` order, so a threshold can be rechecked later without retraining.
- `evaluation/results/raw/finetuned_scibert_run3_reconmind_exam.jsonl`: the first line holds the run 3 `labels` list (the order of every `probs` row), the run name, threshold, max length and findings fingerprint. Then one line per finding with `id`, `pred` and `probs`.

In [ ]:
from datetime import datetime, timezone

RESULTS_DIR = os.path.join(REPO_DIR, 'evaluation', 'results')
os.makedirs(os.path.join(RESULTS_DIR, 'raw'), exist_ok=True)
METRICS_PATH = os.path.join(RESULTS_DIR, 'finetuned_scibert_{r}_metrics.json'.format(r=RUN_NAME))
RAW_PATH = os.path.join(RESULTS_DIR, 'raw', 'finetuned_scibert_{r}.jsonl'.format(r=RUN_NAME))
EXAM_PATH = os.path.join(RESULTS_DIR, 'raw',
                         'finetuned_scibert_{r}_reconmind_exam.jsonl'.format(r=RUN_NAME))

out = {
    'model_kind': 'finetuned',
    'provider': 'huggingface',
    'model': MODEL_NAME,
    'run_name': RUN_NAME,
    'batch_size': BATCH_SIZE,
    'eval_sha256': EVAL_SHA256,
    'failed_sentences': 0,
    'written': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'epochs': EPOCHS,
    'epochs_run': EPOCHS_RUN,
    'stopped_early': STOPPED_EARLY,
    'patience': PATIENCE,
    'best_epoch': best_epoch,
    'best_val_micro_f1': VAL_F1,
    'threshold': THRESHOLD,
    'max_length': MAX_LENGTH,
    'pct_truncated_at_256': round(100 * CUT_AT_256, 3),
    'train_seconds': TRAIN_SECONDS,
    'seed': SEED,
    'learning_rate': LEARNING_RATE,
    'lr_schedule': 'linear warmup then linear decay to zero',
    'warmup_fraction': WARMUP_FRACTION,
    'warmup_steps': WARMUP_STEPS,
    'total_steps': TOTAL_STEPS,
    'loss': 'BCEWithLogitsLoss with pos_weight',
    'pos_weight_cap': POS_WEIGHT_CAP,
    'pos_weight_counted_on': 'train',
    'pos_weight': {lab: round(w, 4) for lab, w in zip(LABELS, POS_WEIGHT)},
    'fp16': True,
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'gpu': GPU_NAME,
    'code_commit': CODE_COMMIT,
    'n_labels': N_LABELS,
    'tram50_not_in_label_set': TRAM_NOT_IN_LABELS,
    'training_data': {
        'drive_dir': RUN3_DATA_DIR,
        'sha256': RUN3_SHA256,
        'checked_against': 'evaluation/results/run3_data_manifest.json',
        'train_rows': len(train_rows),
        'val_rows': len(val_rows),
    },
    'threshold_chosen_on': 'run 3 val split',
    'raw_file': os.path.relpath(RAW_PATH, REPO_DIR),
    'reconmind_exam': {
        'findings_sha256': FINDINGS_SHA256,
        'n_findings': len(findings),
        'raw_file': os.path.relpath(EXAM_PATH, REPO_DIR),
    },
    'epoch_history': epoch_history,
    'val_threshold_sweep': val_sweep,
    'scorings': {
        'tram50_filtered': 'predictions limited to the 50 TRAM IDs, label set is the 50; '
                           'comparable to run 2 and Model A',
        'unfiltered': 'all predictions, label set is the 50 TRAM IDs, so a predicted ID '
                      'outside the 50 counts as out of set',
    },
    'metrics': results,
}
with open(METRICS_PATH, 'w') as fh:
    json.dump(out, fh, indent=1)
    fh.write('\n')

with open(RAW_PATH, 'w') as fh:
    for r, pred, probs in zip(eval_rows, eval_preds, eval_probs):
        fh.write(json.dumps({'id': r['id'], 'gold': r['labels'], 'pred': pred,
                             'probs': [round(p, 4) for p in probs]}, sort_keys=True) + '\n')

with open(EXAM_PATH, 'w') as fh:
    fh.write(json.dumps({'labels': LABELS, 'run_name': RUN_NAME, 'threshold': THRESHOLD,
                         'max_length': MAX_LENGTH, 'findings_sha256': FINDINGS_SHA256}) + '\n')
    for f, pred, probs in zip(findings, exam_preds, exam_probs):
        fh.write(json.dumps({'id': f['id'], 'pred': pred,
                             'probs': [round(p, 4) for p in probs]}) + '\n')

print('metrics:', METRICS_PATH)
print('raw:    ', RAW_PATH)
print('exam:   ', EXAM_PATH)

## 19. Save the model to Google Drive

Colab deletes everything when the session ends, so the trained model and tokenizer are saved to `MyDrive/ReconMind/scibert_attack_mapper_run3`, a new folder named from `RUN_NAME`. Run 1's and run 2's folders are left alone. The same folder gets the run 3 `labels.json` (the exact bytes checked in section 4), the metrics file, and both raw files, since `evaluation/results/raw/` is not committed and would otherwise be lost.

In [ ]:
import shutil

SAVE_DIR = '/content/drive/MyDrive/ReconMind/scibert_attack_mapper_{r}'.format(r=RUN_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(os.path.join(SAVE_DIR, 'labels.json'), 'wb') as fh:
    fh.write(run3_raw['labels.json'])
for path in (METRICS_PATH, RAW_PATH, EXAM_PATH):
    shutil.copy(path, SAVE_DIR)
print('saved to', SAVE_DIR)

## 20. Download the metrics file and the ReconMind predictions

Offers two downloads. Your browser may ask once whether this page may download several files; allow it.

- Put `finetuned_scibert_run3_metrics.json` in `evaluation/results/` of your local ReconMind copy and commit it next to the other metrics files.
- Put `finetuned_scibert_run3_reconmind_exam.jsonl` in `evaluation/results/raw/`, then score it from the repo root with:

```
python3 evaluation/scripts/score_reconmind_exam.py --model-b-run run3 --skip-model-a
```

In [ ]:
from google.colab import files

files.download(METRICS_PATH)
files.download(EXAM_PATH)